# Week 7 Thursday - Asynchronous videos

## Learning

[3Blue1Brown video](https://www.youtube.com/watch?v=IHZwWFHWa-w) on gradient descent.  Recommended clips.

* [0:25-1:24](https://youtu.be/IHZwWFHWa-w?si=Mr29oTledAbooqNT&t=25)
* [3:18-4:05](https://youtu.be/IHZwWFHWa-w?si=VjLDxWp93Jzw8G2u&t=198)
* [5:15-8:16](https://youtu.be/IHZwWFHWa-w?si=854I6w8FXce1DQvG&t=315)

[3Blue1Brown video](https://www.youtube.com/watch?v=Ilg3gGewQ5U) on the intuition behind backpropagation.  Recommended clips.

* [1:42-2:42](https://youtu.be/Ilg3gGewQ5U?si=fPpzsGnMJ1eRANeE&t=102)
* [3:45-5:01](https://youtu.be/Ilg3gGewQ5U?si=fPpzsGnMJ1eRANeE&t=225)
* [9:34-10:10](https://youtu.be/Ilg3gGewQ5U?si=7O1LgWhdzz2hQaQ9&t=574)

## Cross-validation

Here we'll see a situation where using cross-validation is helpful.

Let's fit a logistic regression model to the scikit-learn handwritten digits data.  So far we have typically used `LogisticRegression()` without passing any arguments, but this time let's set an explicit regularization value `LogisticRegression(C=...)`, and let's try to decide what value of this parameter works best.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

In [ ]:
digits = load_digits()
X = digits.data
X_rescale = X / X.max()
y = digits.target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_rescale, 
    y, 
    test_size=0.25, 
    random_state=210
)

Smaller values of `C` mean more regularization, i.e., more prioritization of smaller coefficients.

In [ ]:
c_list = [0.001, 0.01, 0.1, 1, 10, 100, np.inf]

In [ ]:
clf = LogisticRegression(C=1)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
log_loss(y_train, clf.predict_proba(X_train), labels=clf.classes_)

In [ ]:
for c in c_list:
    clf = LogisticRegression(C=c)
    clf.fit(X_train, y_train)
    loss = log_loss(y_train, clf.predict_proba(X_train), labels=clf.classes_)
    print(c)
    print(loss)

In [ ]:
for c in c_list:
    clf = LogisticRegression(C=c)
    clf.fit(X_train, y_train)
    loss = log_loss(y_test, clf.predict_proba(X_test), labels=clf.classes_)
    print(c)
    print(loss)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import get_scorer_names

In [ ]:
get_scorer_names()[10:30]

In [ ]:
cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=210
)

for c in c_list:
    scores = cross_val_score(
        LogisticRegression(C=c),
        X_train,
        y_train,
        cv=cv,
        scoring="neg_log_loss"
    )

    print(c)
    print(-scores.mean())

## Grid Search

Here we'll see how to automate much of the process above using a grid search.

In [ ]:
cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=210
)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
clf = LogisticRegression()

search = GridSearchCV(
    clf,
    param_grid={"C":c_list},
    cv=cv,
    scoring="neg_log_loss"
)

search.fit(X_train, y_train)

In [ ]:
search.best_params_

In [ ]:
len(c_list)

In [ ]:
clf = LogisticRegression()

search = GridSearchCV(
    clf,
    param_grid={"C":c_list, "max_iter":[2, 100, 1000, 3000]},
    cv=cv,
    scoring="neg_log_loss"
)

The previous `search.fit(X_train, y_train)` shown in the video in the next cell gets wrapped by some extra code.  The effect is that we don't see the many warnings from the (intentionally) low max iterations number.  Conceptually this is the same as what's shown in the video, just with the warnings hidden.  Another option is just to clear the cell output from that particular cell, but that would need to be done every time.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    search.fit(X_train, y_train)

In [ ]:
search.best_params_

## f-strings

I want to briefly introduce f-strings, which would have been natural to use in our for loop above.  These were introduced in Python 3.6.

In [ ]:
for c in c_list:
    clf = LogisticRegression(C=c)
    clf.fit(X_train, y_train)
    loss = log_loss(y_test, clf.predict_proba(X_test), labels=clf.classes_)
    print(f"C: {c:>6g}   Loss: {loss:>.4f}")

## StandardScaler and Pipeline

Here we'll see two more useful classes from scikit-learn, `StandardScaler` and `Pipeline`.

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
df = sns.load_dataset("mpg").dropna(axis=0)

In [ ]:
df.dtypes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[["horsepower", "weight"]], 
    df["mpg"], 
    test_size=0.25, 
    random_state=210
)

In [ ]:
reg = LinearRegression()

In [ ]:
reg.fit(X_train, y_train)

In [ ]:
reg.coef_

In [ ]:
reg.feature_names_in_

In [ ]:
X_train.describe()

Here are two possible reasons we might want to rescale columns.
* Some models, including neural networks, are easier to train on data that has been rescaled to have approximately mean 0 and standard deviation 1;
* For models like linear regression and logistic regression, having all columns on approximately the same scale makes the magnitudes of the coefficients easier to compare.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
X_train_scaled.describe()

In [ ]:
X_train_scaled.mean(axis=0)

In [ ]:
X_train_scaled.std(axis=0)

In [ ]:
reg.fit(X_train_scaled, y_train)

In [ ]:
reg.coef_

In [ ]:
y_test[:3]

In [ ]:
reg.predict(X_test[:3])

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]
)

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
pipe.predict(X_test[:3])

In [ ]:
pipe.coef_

In [ ]:
pipe.named_steps

In [ ]:
type(pipe.named_steps)

In [ ]:
pipe.named_steps["model"]

In [ ]:
pipe.named_steps["model"].coef_

## OneHotEncoder

Here we'll see a way to include a non-numeric column in our Linear Regression model.

In [ ]:
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

In [ ]:
df = sns.load_dataset("mpg").dropna(axis=0)

In [ ]:
df["origin"].value_counts()